# Analisi del Progetto UNet Universitario — RF02 / RF03 / RF04

Questo notebook analizza il mini-progetto UNet universitario (segmentazione Carvana)
in relazione al progetto di low-light enhancement.

| Task | Notebook |
|---|---|
| **RF02** | Confronto architettura universitaria vs baseline progetto |
| RF03 | Componenti riutilizzabili |
| RF04 | Analisi mancanza di generalizzazione cross-dataset |

---
## RF02 — Confronto architettura universitaria vs baseline low-light

Obiettivo: identificare le differenze chiave tra la UNet universitaria (segmentazione)
e la UNet baseline che useremo per low-light enhancement.

In [ ]:
import sys
from pathlib import Path

# Aggiunge la root del progetto al path
PROJECT_ROOT = Path("/kaggle/input/low-light-repo") if Path("/kaggle").exists() else Path("..")
sys.path.insert(0, str(PROJECT_ROOT))

# Aggiunge il mini-progetto universitario al path per importarlo
UNET_REF = PROJECT_ROOT / "references" / "university_unet_project"
sys.path.insert(0, str(UNET_REF))

import torch
import torch.nn as nn
from unet.unet_model import UNet as UNetRef

print("Import OK")

### 1. Differenza di task

| Aspetto | UNet universitaria | Baseline low-light |
|---|---|---|
| **Task** | Segmentazione semantica | Image enhancement (restoration) |
| **Input** | Immagine RGB `(3, H, W)` | Immagine low-light RGB `(3, H, W)` |
| **Output** | Mappa di probabilità `(n_classes, H, W)` | Immagine enhanced RGB `(3, H, W)` |
| **Supervisione** | Maschera discreta (etichette per pixel) | Immagine ground truth continua `[0, 1]` |
| **Dataset** | Carvana (segmentazione auto) | LOL-v2 (coppie low/normal) |

Il cambio di task è la differenza più profonda: la segmentazione produce
una classificazione *discreta* per pixel, mentre enhancement produce valori
*continui* nell'intervallo [0, 1].

### 2. Differenze architetturali

| Componente | UNet universitaria | Baseline low-light |
|---|---|---|
| **Encoder** | 4 Down: 64→128→256→512→1024 | Identico |
| **Bottleneck** | DoubleConv a 1024 canali | Identico |
| **Decoder** | 4 Up con skip connections | Identico |
| **Output head** | `Conv1×1(64 → n_classes)` | `Conv1×1(64 → 3)` + `Sigmoid` |
| **Attivazione finale** | Softmax/Sigmoid (applicata nella loss) | **Sigmoid** — forza output in [0, 1] |
| **Upsampling** | `ConvTranspose2d` (default) o bilinear | Stessa scelta |
| **Skip connections** | `cat` lungo dim=1 + padding dinamico | Identico |

**Unica modifica strutturale necessaria**: cambiare l'output head da
`Conv1×1(64 → n_classes)` a `Conv1×1(64 → 3)` seguito da `Sigmoid`.

> **Perché Sigmoid e non clamp?**  
> `clamp(0, 1)` è non-derivabile agli estremi: il gradiente è zero fuori
> dall'intervallo, il che può bloccare l'apprendimento nei pixel saturi.
> `Sigmoid` mantiene gradienti non-zero ovunque.

### 3. Differenze nella loss

| Aspetto | UNet universitaria | Baseline low-light |
|---|---|---|
| **Loss primaria** | `CrossEntropyLoss` | `L1Loss` (MAE) |
| **Loss secondaria** | `DiceLoss` | `SSIMLoss` |
| **Target** | Etichette intere `[0, n_classes)` | Immagine float `[0.0, 1.0]` |

**Perché non CrossEntropy + Dice per l'enhancement?**  
CrossEntropy richiede target discreti (classi). SSIM misura la similarità
strutturale tra immagini continue, catturando contrasto e struttura locale
che L1 da solo non coglie.

### 4. Differenze nella pipeline di training

| Aspetto | UNet universitaria | Baseline low-light |
|---|---|---|
| **Split train/val** | `random_split` sull'intero dataset | Solo dalla cartella `Train/` ufficiale |
| **Augmentation** | Nessuna esplicita | Flip + crop + ColorJitter (solo su low) |
| **Augmentation accoppiata** | N/A | Obbligatoria: flip/crop identici per low e normal |
| **Ottimizzatore** | RMSprop (lr=1e-5, momentum=0.999) | Adam/AdamW, lr da configurazione |
| **Scheduler** | ReduceLROnPlateau su Dice (max) | ReduceLROnPlateau su validation loss o PSNR |
| **Metrica val** | Dice score | PSNR |
| **Logging** | WandB | TensorBoard (`TrainingLogger`) |

> **Nota sul random_split**: nel mini-progetto train e val prendono dati
> dallo stesso pool. Per LOL-v2 questo sarebbe scorretto: il Test set ufficiale
> deve rimanere completamente separato, e il val va estratto solo dal Train ufficiale.

In [ ]:
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# UNet universitaria: segmentazione (n_classes=2, bilinear=False)
unet_seg = UNetRef(n_channels=3, n_classes=2, bilinear=False)
total_seg, _ = count_params(unet_seg)

# UNet baseline low-light: output head cambiato a 3 canali
unet_ll = UNetRef(n_channels=3, n_classes=3, bilinear=False)
total_ll, _ = count_params(unet_ll)

print(f"UNet segmentazione (n_classes=2) : {total_seg:>10,} parametri")
print(f"UNet low-light     (n_classes=3) : {total_ll:>10,} parametri")
print(f"Differenza                       : {total_ll - total_seg:>+10,} (solo Conv1x1 output)")
print()
print("La differenza e' minima: cambia solo l'ultimo strato Conv1x1.")

### Sintesi RF02

| Categoria | Entità della modifica |
|---|---|
| Encoder (inc + down1-4) | **Nessuna** |
| Decoder (up1-4) | **Nessuna** |
| Output head | **Minima**: `n_classes=2` → `n_classes=3` + `Sigmoid` |
| Loss | **Completa**: CrossEntropy+Dice → L1+SSIM |
| Metrica | **Completa**: Dice → PSNR/SSIM |
| Dataset pipeline | **Significativa**: paired augmentation, split ufficiale |
| Logging | **Medio**: WandB → TensorBoard |

La UNet universitaria è un **punto di partenza diretto** per la baseline:
encoder e decoder si riutilizzano invariati, e le modifiche si concentrano
sull'output head e sulla pipeline di training.

---
## RF03 — Componenti riutilizzabili dal mini-progetto

Obiettivo: identificare blocchi e scelte progettuali del mini-progetto
che possono essere copiati o adattati direttamente nel progetto low-light,
evitando di riscrivere da zero ciò che già funziona.

### 1. Blocchi architetturali — riuso come base

I tre blocchi fondamentali dell'encoder/decoder sono architetturalmente
equivalenti a quelli che servono per il progetto low-light e possono essere
usati come base diretta in `src/models/unet_parts.py`, adattandoli allo stile
e alle convenzioni del nuovo progetto.

#### `DoubleConv`
```python
Conv2d(3×3, padding=1, bias=False) → BN → ReLU
Conv2d(3×3, padding=1, bias=False) → BN → ReLU
```
Riuso: **alto** — adattabile quasi invariato.  
Motivo: è il blocco convolutivo standard di UNet, indipendente dal task.

> **Nota su BatchNorm**: BN è mantenuto nella baseline per coerenza con la
> UNet di riferimento. Se il batch size effettivo dovesse essere molto piccolo
> (es. bs=1 o bs=2), BN diventa instabile perché le statistiche per-batch
> sono poco rappresentative. In quel caso si può considerare GroupNorm o
> InstanceNorm come alternativa.

#### `Down`
```python
MaxPool2d(2) → DoubleConv
```
Riuso: **alto** — adattabile quasi invariato.  
Motivo: il downsampling con MaxPool è deterministico e task-agnostico.

#### `Up`
```python
ConvTranspose2d(kernel=2, stride=2)  # oppure Upsample bilinear
→ F.pad dinamico per dimensioni dispari
→ cat(skip, upsampled)
→ DoubleConv
```
Riuso: **alto** — adattabile quasi invariato.  
Il padding dinamico (`diffY`, `diffX`) gestisce già input di dimensioni
non divisibili per 2\^depth, che si verifica con immagini reali LOL-v2.

### 2. Schema canali encoder

```
3 → 64 → 128 → 256 → 512 → 1024
```

Riuso: **come punto di partenza**.  
Con immagini 256×256 e batch size 8 su T4 (16 GB VRAM), il bottleneck
a 1024 canali è generalmente sostenibile. Se la VRAM dovesse scarseggiare,
si può scalare a `3 → 32 → 64 → 128 → 256 → 512` (metà canali).

| Configurazione | Parametri approx | VRAM stimata (bs=8, 256²) |
|---|---|---|
| Canali ×1 (originale) | ~31 M | ~10–12 GB |
| Canali ×0.5 | ~8 M | ~3–4 GB |

> **Nota**: i valori di parametri e VRAM sono stime indicative, non valori
> garantiti. Il consumo reale dipende da batch size, AMP, dettagli
> implementativi e hardware. Verificare sempre con un profiler dopo
> aver implementato il modello reale.

In [ ]:
# Verifica parametri per le due configurazioni di canali
unet_full = UNetRef(n_channels=3, n_classes=3, bilinear=False)
total_full = count_params(unet_full)

# Versione ridotta: approssimazione con DoubleConv semplificati
# NOTA: questo conteggio è solo una stima indicativa. Il numero esatto
# di parametri va ricalcolato dopo aver implementato il modello baseline reale.
import torch.nn as nn

class DoubleConvSmall(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

# UNet con canali dimezzati (approssimazione — non include ConvTranspose2d del decoder)
half_ch = [32, 64, 128, 256, 512]  # invece di 64,128,256,512,1024
layers_half = []
ch_in = 3
for ch_out in half_ch:
    layers_half.append(DoubleConvSmall(ch_in, ch_out))
    ch_in = ch_out
# Decoder speculare + output
for ch_out in reversed(half_ch[:-1]):
    layers_half.append(DoubleConvSmall(ch_in + ch_out, ch_out))
    ch_in = ch_out
layers_half.append(nn.Conv2d(ch_in, 3, 1))
unet_half = nn.Sequential(*layers_half)
total_half = count_params(unet_half)

print(f'UNet canali x1 (64→1024) : {total_full:>10,} parametri')
print(f'UNet canali x0.5 (32→512): {total_half:>10,} parametri (stima)')
print(f'Riduzione stimata        : {1 - total_half/total_full:.0%}')
print()
print('Nota: il conteggio x0.5 e\' una stima approssimativa.')
print('Ricalcolare dopo aver implementato il modello baseline reale.')

### 3. Gradient clipping

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

Riuso: **alto**, valore `max_norm=1.0` come punto di partenza.  
Motivo: il gradient clipping previene esplosioni del gradiente comuni
nei layer profondi di UNet durante le prime epoche. Con loss L1+SSIM
i gradienti sono generalmente più stabili che con CrossEntropy, ma
il clipping rimane una buona pratica a costo zero.

### 4. Automatic Mixed Precision (AMP)

```python
# Setup
scaler = torch.cuda.amp.GradScaler(enabled=amp)

# Training step
with torch.autocast(device_type='cuda', enabled=amp):
    output = model(low)
    loss = criterion(output, normal)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)
clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()
```

Riuso: **alto** — pattern riutilizzabile come base, con un adattamento:  
- Nel mini-progetto: `device.type` passato ad `autocast` — corretto.  
- Nel progetto low-light: stesso pattern, da abilitare su Kaggle T4
  dove AMP dimezza il consumo VRAM e accelera il training del ~30–40%.

> `scaler.unscale_` va chiamato **prima** di `clip_grad_norm_`,
> altrimenti il clipping opera sui gradienti scalati (valori errati).

### 5. Learning rate scheduler

```python
# Mini-progetto (Dice score, da massimizzare)
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=5)

# Progetto low-light — due opzioni a seconda della metrica monitorata:
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5)  # validation loss
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=5)  # PSNR su val
```

Riuso: **alto** — logica identica, parametri da adattare alla metrica scelta.  
La scelta del `mode` dipende da cosa si decide di monitorare:
- `mode='min'` se si monitora la **validation loss** (L1 o L1+SSIM) — da minimizzare;
- `mode='max'` se si monitora il **PSNR** su validation — da massimizzare.

`patience=5` è un valore ragionevole come punto di partenza per entrambe le opzioni.

### Sintesi RF03 — Tabella riuso

| Componente | File sorgente | Riuso | Note |
|---|---|---|---|
| `DoubleConv` | `unet/unet_parts.py` | Riuso alto | → `src/models/unet_parts.py`; valutare GroupNorm/InstanceNorm se bs molto piccolo |
| `Down` | `unet/unet_parts.py` | Riuso alto | → `src/models/unet_parts.py` |
| `Up` | `unet/unet_parts.py` | Riuso alto | → `src/models/unet_parts.py` |
| Schema canali 64→1024 | `unet/unet_model.py` | Punto di partenza | Valutare ×0.5 se VRAM insufficiente; stime indicative |
| Gradient clipping | `train.py:143` | Riuso alto | `max_norm=1.0` come punto di partenza |
| AMP (GradScaler + autocast) | `train.py:127,141-145` | Riuso alto | Abilitare su Kaggle T4 |
| ReduceLROnPlateau | `train.py:110` | Riuso alto | `mode='min'` su val loss o `mode='max'` su PSNR; `patience=5` |
| Output head | `unet/unet_parts.py:OutConv` | Adattare | `n_classes=3` + `Sigmoid` |

---
## RF04 — Perché la UNet universitaria non generalizzerebbe cross-dataset

Obiettivo: analizzare i fattori strutturali e progettuali che limiterebbero
la generalizzazione cross-dataset della UNet universitaria se applicata
direttamente al problema low-light enhancement.

Per *cross-dataset* si intende: training su LOL-v2, valutazione su LOL-v1
e/o ExDark — dataset con distribuzioni di degradazione, scene e condizioni
di acquisizione diverse dal training set.

### 1. Mismatch di task (causa strutturale)

La UNet universitaria è addestrata per **segmentazione binaria** (Carvana).
Il suo output è una mappa di classi discrete, supervisionata da maschere binarie.
Non ha mai visto coppie low/normal-light e non modella alcuna trasformazione
di illuminazione.

Questa è la causa più fondamentale: **un modello di segmentazione non è
in alcun modo trasferibile direttamente a enhancement**, indipendentemente
dal dataset. Non si tratta di mancata generalizzazione cross-dataset, ma
di incompatibilità di task.

> Nelle sezioni seguenti si analizza cosa succederebbe a una UNet con la
> stessa architettura ma *ri-addestrata* per low-light enhancement su LOL-v2:
> quali scelte del mini-progetto ne limiterebbero la generalizzazione.

### 2. Domain shift tra i dataset

I tre dataset usati nel progetto hanno distribuzioni di degradazione
significativamente diverse:

| Dataset | Tipo di scene | Sorgente di degradazione | Ground truth |
|---|---|---|---|
| LOL-v2 Real | Interno, controllato | Esposizione ridotta, rumore del sensore | Accoppiata (stessa scena) |
| LOL-v2 Synthetic | Sintetico | Degradazione simulata (gamma, rumore) | Accoppiata |
| LOL-v1 | Interno, stili diversi | Esposizione ridotta, diverso hardware | Accoppiata |
| ExDark | Esterno/interno, naturale | Illuminazione reale scarsa, no ground truth | Nessuna |

Un modello addestrato su LOL-v2 impara a correggere lo specifico profilo
di degradazione di quel dataset: la distribuzione di luminosità, il tipo
di rumore, e la struttura spaziale delle scene indoor di LOL-v2.

**Conseguenza per la generalizzazione**:
- Su **LOL-v1**: le scene e il hardware di acquisizione sono diversi;
  il modello può produrre artefatti o sotto-correggere zone con
  distribuzione di rumore diversa da quella vista in training.
- Su **ExDark**: scene outdoor con illuminazione naturale scarsa;
  distribuzione completamente diversa da LOL-v2 indoor.
  Inoltre non esiste ground truth: la valutazione è solo no-reference
  (NIQE, BRISQUE), il che rende impossibile quantificare PSNR/SSIM.

### 3. Statistiche BatchNorm legate al training set

Ogni `DoubleConv` usa `BatchNorm2d`, che durante il training accumula
le statistiche *running mean* e *running variance* del dataset di training.
In modalità `eval()`, BN normalizza le feature map con queste statistiche
fisse — che rispecchiano la distribuzione di LOL-v2.

Se le feature map di un'immagine LOL-v1 o ExDark hanno una distribuzione
significativamente diversa, la normalizzazione batch introduce uno shift
sistematico che degrada le predizioni.

**Alternativa considerata** (vedi RF03): GroupNorm o InstanceNorm non
accumulano statistiche di dataset — normalizzano per canale o per
istanza, rendendosi naturalmente più robuste al domain shift.

| Normalizzazione | Dipende dal dataset di training? | Note |
|---|---|---|
| `BatchNorm2d` | Sì (running stats) | Ottima con bs grande, fragile cross-dataset |
| `GroupNorm` | No | Stabile anche con bs piccolo, più robusto cross-dataset |
| `InstanceNorm2d` | No | Per-immagine; usato in style transfer |

> Il mini-progetto usa BatchNorm senza discutere questa limitazione.
> Per il progetto low-light, dove la generalizzazione cross-dataset
> è un obiettivo esplicito, vale la pena valutare GroupNorm nell'architettura
> attention (variante avanzata).

### 4. Assenza di augmentation — overfitting all'aspetto del training set

Il mini-progetto non applica nessuna augmentation geometrica o fotometrica
durante il training (`train.py` non ha trasformazioni casuali).

Conseguenze:
- Il modello si adatta alle specifiche pose, orientamenti e condizioni
  di illuminazione viste in LOL-v2.
- Scene di LOL-v1 o ExDark con orientamenti o illuminazioni diverse
  cadono fuori dalla distribuzione appresa.

**Nel progetto low-light** questo problema è già affrontato: `PairedAugmentation`
applica flip orizzontale, random crop e ColorJitter (solo su low), aumentando
la variabilità di training e riducendo l'overfitting alla distribuzione
specifica di LOL-v2.

### 5. Split non riproducibile e data leakage potenziale

Il mini-progetto usa `random_split` sull'intero dataset con un seed fisso.
Applicato a LOL-v2, questo causerebbe due problemi:

1. **Violazione dello split ufficiale**: LOL-v2 ha cartelle `Train/` e `Test/`
   ufficiali. Mischiare tutti i file e ridividerli casualmente contamina
   il test set con scene viste durante il training.

2. **Metriche ottimistiche**: se scene del Test/ finiscono nel training
   (anche parzialmente), PSNR e SSIM sul test set risultano artificialmente
   alti — non riflettono la vera capacità di generalizzazione.

**Nel progetto low-light** questo è già risolto: `create_and_save_train_val_split`
opera *solo* sugli stem della cartella `Train/` ufficiale, e il test set
viene gestito separatamente con `save_test_split` che non mescola mai
i due pool.

### 6. Nessun meccanismo di adattamento al dominio

La UNet universitaria è un'architettura feed-forward pura: dato un input,
produce un output senza alcuna capacità di adattarsi alle statistiche
specifiche dell'immagine in ingresso.

Per la generalizzazione cross-dataset, meccanismi come:
- **Attention gates** (variante avanzata del progetto): pesano le
  skip connections in base al contenuto locale, riducendo la dipendenza
  dalle statistiche globali del training set;
- **InstanceNorm** nelle prime fasi del decoder: normalizza per immagine
  invece che per dataset;

possono migliorare la robustezza cross-dataset rispetto a una UNet pura.

> Questa è la motivazione principale per implementare una **variante attention**
> oltre alla baseline: non solo aumenta la capacità del modello, ma introduce
> un meccanismo che in letteratura mostra migliore generalizzazione
> su scene non viste durante il training.

### Sintesi RF04

| Causa | Presente nel mini-progetto | Impatto sulla generalizzazione | Mitigazione nel progetto |
|---|---|---|---|
| Mismatch di task | Sì (segmentazione) | Totale — non trasferibile | Ri-addestrare per enhancement |
| Domain shift dataset | Intrinseco al problema | Alto | Valutazione cross-dataset esplicita |
| BatchNorm con running stats | Sì | Medio-alto | Valutare GroupNorm nella variante attention |
| Assenza di augmentation | Sì | Medio | `PairedAugmentation` già implementata |
| Split non ufficiale | Sì (`random_split`) | Medio (metriche ottimistiche) | `create_and_save_train_val_split` su Train/ |
| Nessun meccanismo di adattamento | Sì | Medio | Attention gates nella variante avanzata |

**Conclusione**: la UNet universitaria presenta limitazioni strutturali e
progettuali che ne ridurrebbero la generalizzazione cross-dataset.
Il progetto low-light affronta esplicitamente quattro di questi sei
fattori già nelle scelte di implementazione della pipeline dati e
dell'architettura. Il domain shift intrinseco tra dataset rimane
il limite principale e sarà quantificato nella fase di valutazione (E-task).